- Given employee swipe in swipe out data. For each employee find the total in time.

In [0]:
import datetime
from pyspark.sql.types import StructType, StructField, TimestampType, LongType, StringType
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
spark = SparkSession.builder \
    .appName("TotalInTime") \
    .getOrCreate()

In [0]:
input_data = [
    (11114, datetime.datetime.strptime('08:30:00.00', "%H:%M:%S.%f"), "I"),
    (11114, datetime.datetime.strptime('10:30:00.00', "%H:%M:%S.%f"), 'O'),
    (11114, datetime.datetime.strptime('11:30:00.00', "%H:%M:%S.%f"), 'I'),
    (11114, datetime.datetime.strptime('15:30:00.00', "%H:%M:%S.%f"), 'O'),
    (11115, datetime.datetime.strptime('09:30:00.00', "%H:%M:%S.%f"), 'I'),
    (11115, datetime.datetime.strptime('17:30:00.00', "%H:%M:%S.%f"), 'O')
]

In [0]:
input_schema = StructType([
    StructField('emp_id', LongType(), True),
    StructField('punch_time', TimestampType(), True),
    StructField('flag', StringType(), True)
])


In [0]:
df = spark.createDataFrame(data=input_data, schema=input_schema)


In [0]:
window_def = Window.partitionBy('emp_id').orderBy(col('punch_time'))


In [0]:
df = df.withColumn('prev_time', lag(col('punch_time')).over(window_def))


In [0]:
df = df.withColumn('time_diff', (col('punch_time').cast('long') - col('prev_time').cast('long'))/3600)


In [0]:
df = df.groupBy('emp_id').agg(sum(when(col('flag') == 'O', col('time_diff')).otherwise(0)).alias('total_time'))


In [0]:
df.show()


+------+----------+
|emp_id|total_time|
+------+----------+
| 11114|       6.0|
| 11115|       8.0|
+------+----------+

